In [1]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules

c:\Users\15195\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Read the transaction item table and cluster results.
transaction_product = pd.read_csv(
    r"C:\Users\15195\Desktop\coding part\dataset merged\transaction_product_merged.csv"
)

clustered_households = pd.read_csv(
    r"C:\Users\15195\Desktop\coding part\k-means-修改版\clustered_household_features (k=3).csv"
)

print("Transaction-product shape:", transaction_product.shape)
print("Clustered households shape:", clustered_households.shape)

print(transaction_product.columns.tolist())
print(clustered_households.columns.tolist())

Transaction-product shape: (2595732, 18)
Clustered households shape: (2500, 19)
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']
['household_key', 'recency', 'frequency', 'monetary', 'total_quantity', 'unique_products', 'category_diversity', 'first_purchase_day', 'last_purchase_day', 'average_basket_value', 'average_basket_size', 'customer_lifetime_days', 'total_retail_discount', 'total_coupon_discount', 'total_coupon_match_discount', 'total_discount', 'discount_ratio', 'coupon_discount_ratio', 'cluster']


In [3]:
# Extract the cluster labels for households
cluster_labels = (
    clustered_households[
        ["household_key", "cluster"]
    ]
    .drop_duplicates(subset=["household_key"])
)

print(cluster_labels.head())

print(
    cluster_labels["cluster"]
    .value_counts()
    .sort_index()
)

   household_key  cluster
0              1        1
1              2        1
2              3        1
3              4        2
4              5        2
cluster
0     286
1    1660
2     554
Name: count, dtype: int64


In [4]:
# Merge the clustering labels back into the transaction product table
transaction_product_with_segment = transaction_product.merge(
    cluster_labels,
    on="household_key",
    how="inner",
    validate="many_to_one"
)

print("Merged shape:", transaction_product_with_segment.shape)

print(
    transaction_product_with_segment[
        [
            "household_key",
            "BASKET_ID",
            "PRODUCT_ID",
            "COMMODITY_DESC",
            "cluster"
        ]
    ].head()
)

print(
    transaction_product_with_segment["cluster"]
    .value_counts()
    .sort_index()
)



Merged shape: (2595732, 19)
   household_key    BASKET_ID  PRODUCT_ID               COMMODITY_DESC  \
0           2375  26984851472     1004906                     POTATOES   
1           2375  26984851472     1033142                       ONIONS   
2           2375  26984851472     1036325      VEGETABLES - ALL OTHERS   
3           2375  26984851472     1082185               TROPICAL FRUIT   
4           2375  26984851472     8160430  ORGANICS FRUIT & VEGETABLES   

   cluster  
0        1  
1        1  
2        1  
3        1  
4        1  
cluster
0     723767
1    1787219
2      84746
Name: count, dtype: int64


In [5]:
# Save the transaction_product_with_segment table
transaction_product_with_segment.to_csv(
    r"C:\Users\15195\Desktop\transaction_product_with_segment.csv",
    index=False
)

print("Saved transaction_product_with_segment.csv")

Saved transaction_product_with_segment.csv


In [6]:
# Clean up the shopping cart data
# The purpose of the cleanup is to aim to retain: valid BASKET_ID, valid COMMODITY_DESC, valid cluster, positive sales amount, positive purchase quantity
basket_source = transaction_product_with_segment.copy()

basket_source = basket_source.dropna(
    subset=[
        "household_key",
        "BASKET_ID",
        "COMMODITY_DESC",
        "cluster"
    ]
)

basket_source = basket_source[
    (basket_source["SALES_VALUE"] > 0) &
    (basket_source["QUANTITY"] > 0)
].copy()

basket_source["COMMODITY_DESC"] = (
    basket_source["COMMODITY_DESC"]
    .astype(str)
    .str.strip()
    .str.upper()
)

basket_source["cluster"] = (
    basket_source["cluster"]
    .astype(int)
)

print("Valid baskets:", basket_source["BASKET_ID"].nunique())
print("Commodity categories:", basket_source["COMMODITY_DESC"].nunique())
print("Clusters:", sorted(basket_source["cluster"].unique()))

Valid baskets: 275539
Commodity categories: 307
Clusters: [np.int64(0), np.int64(1), np.int64(2)]


In [7]:
# Create a comprehensive basket_items.csv file 
# Each row represents a specific product category within a shopping basket
basket_items = (
    basket_source[
        [
            "household_key",
            "BASKET_ID",
            "COMMODITY_DESC"
        ]
    ]
    .drop_duplicates()
)

print(basket_items.head())
print("Basket-item rows:", len(basket_items))
print("Number of baskets:", basket_items["BASKET_ID"].nunique())

   household_key    BASKET_ID               COMMODITY_DESC
0           2375  26984851472                     POTATOES
1           2375  26984851472                       ONIONS
2           2375  26984851472      VEGETABLES - ALL OTHERS
3           2375  26984851472               TROPICAL FRUIT
4           2375  26984851472  ORGANICS FRUIT & VEGETABLES
Basket-item rows: 1903959
Number of baskets: 275539


In [8]:
# Save the basket_items.csv table
basket_items.to_csv(
    r"C:\Users\15195\Desktop\basket_items.csv",
    index=False
)

print("Saved basket_items.csv")

Saved basket_items.csv


In [9]:
# Filter out low-frequency product categories
# To prevent the matrix from becoming too large, initially retain categories that appear in at least 1% of the shopping baskets
total_baskets = basket_items["BASKET_ID"].nunique()

commodity_frequency = (
    basket_items
    .groupby("COMMODITY_DESC")["BASKET_ID"]
    .nunique()
    .sort_values(ascending=False)
)

minimum_basket_count = max(
    20,
    int(total_baskets * 0.01)
)

eligible_commodities = (
    commodity_frequency[
        commodity_frequency >= minimum_basket_count
    ]
    .index
)

basket_items_filtered = basket_items[
    basket_items["COMMODITY_DESC"]
    .isin(eligible_commodities)
].copy()

print("Total baskets:", total_baskets)
print("Minimum basket count:", minimum_basket_count)
print(
    "Eligible categories:",
    basket_items_filtered["COMMODITY_DESC"].nunique()
)

Total baskets: 275539
Minimum basket count: 2755
Eligible categories: 150


In [10]:
# The categories remain too numerous, so only the top 100 are included here.
top_commodities = (
    basket_items_filtered["COMMODITY_DESC"]
    .value_counts()
    .head(100)
    .index
)

basket_items_filtered = basket_items_filtered[
    basket_items_filtered["COMMODITY_DESC"]
    .isin(top_commodities)
].copy()

print(
    "Categories after Top-100 filtering:",
    basket_items_filtered["COMMODITY_DESC"].nunique()
)

Categories after Top-100 filtering: 100


In [11]:
# Save the entire basket_matrix.csv file
basket_matrix = (
    basket_items_filtered
    .assign(present=True)
    .pivot_table(
        index="BASKET_ID",
        columns="COMMODITY_DESC",
        values="present",
        aggfunc="max",
        fill_value=False
    )
    .astype(bool)
)

print("Basket matrix shape:", basket_matrix.shape)
print(basket_matrix.head())

Basket matrix shape: (256305, 100)
COMMODITY_DESC  APPLES  BACON  BAG SNACKS  BAKED BREAD/BUNS/ROLLS  \
BASKET_ID                                                           
26984851472      False  False       False                   False   
26984851516      False  False       False                    True   
26984896261      False  False       False                   False   
26984905972      False  False       False                    True   
26984945254      False  False       False                   False   

COMMODITY_DESC  BAKED SWEET GOODS  BAKING MIXES  BAKING NEEDS  BATH TISSUES  \
BASKET_ID                                                                     
26984851472                 False         False         False         False   
26984851516                 False         False         False         False   
26984896261                 False         False         False         False   
26984905972                 False         False         False         False   
2698494

In [12]:
#  Save the entire basket_matrix.csv file
basket_matrix.to_csv(
    r"C:\Users\15195\Desktop\basket_matrix.csv"
)

print("Saved basket_matrix.csv")

Saved basket_matrix.csv


In [13]:
# Run Apriori on the whole dataset
overall_frequent_itemsets = apriori(
    basket_matrix,
    min_support=0.01,
    use_colnames=True,
    max_len=3
)

overall_frequent_itemsets = (
    overall_frequent_itemsets
    .sort_values(
        by="support",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Number of frequent itemsets:",
    len(overall_frequent_itemsets)
)

print(overall_frequent_itemsets.head(20))

Number of frequent itemsets: 2754
     support                                           itemsets
0   0.279140                           frozenset({SOFT DRINKS})
1   0.269983                   frozenset({FLUID MILK PRODUCTS})
2   0.235122                frozenset({BAKED BREAD/BUNS/ROLLS})
3   0.182813                                frozenset({CHEESE})
4   0.163770                            frozenset({BAG SNACKS})
5   0.143169                                  frozenset({BEEF})
6   0.126669                        frozenset({TROPICAL FRUIT})
7   0.120837  frozenset({BAKED BREAD/BUNS/ROLLS, FLUID MILK ...
8   0.108722                                  frozenset({EGGS})
9   0.107622                     frozenset({COUPON/MISC ITEMS})
10  0.101130                frozenset({REFRGRATD JUICES/DRNKS})
11  0.099069      frozenset({FLUID MILK PRODUCTS, SOFT DRINKS})
12  0.098039                           frozenset({COLD CEREAL})
13  0.095788           frozenset({FLUID MILK PRODUCTS, CHEESE})
14  0.

In [14]:
# Save the overall frequent item sets
# Since itemsets are frozensets, they need to be converted to text before being saved
overall_frequent_itemsets_save = (
    overall_frequent_itemsets.copy()
)

overall_frequent_itemsets_save["itemsets"] = (
    overall_frequent_itemsets_save["itemsets"]
    .apply(
        lambda x: " | ".join(
            sorted(list(x))
        )
    )
)

overall_frequent_itemsets_save.to_csv(
    r"C:\Users\15195\Desktop\frequent_itemsets.csv",
    index=False
)

print("Saved frequent_itemsets.csv")

Saved frequent_itemsets.csv


In [15]:
# Generate comprehensive Association Rules
overall_rules = association_rules(
    overall_frequent_itemsets,
    metric="confidence",
    min_threshold=0.10
)

print("Rules before filtering:", len(overall_rules))

Rules before filtering: 8980


In [16]:
# Establish filtering rules
overall_rules_filtered = overall_rules[
    (overall_rules["support"] >= 0.005) &
    (overall_rules["confidence"] >= 0.20) &
    (overall_rules["lift"] > 1.00)
].copy()

# Maintain the one-on-one category rules for ease of understanding
overall_rules_filtered = overall_rules_filtered[
    (overall_rules_filtered["antecedents"].apply(len) == 1) &
    (overall_rules_filtered["consequents"].apply(len) == 1)
].copy()

# Convert the category set to text
overall_rules_filtered["antecedents"] = (
    overall_rules_filtered["antecedents"]
    .apply(
        lambda x: " | ".join(
            sorted(list(x))
        )
    )
)

overall_rules_filtered["consequents"] = (
    overall_rules_filtered["consequents"]
    .apply(
        lambda x: " | ".join(
            sorted(list(x))
        )
    )
)

# Perform the sorting
overall_rules_filtered = (
    overall_rules_filtered
    .sort_values(
        by=[
            "lift",
            "confidence",
            "support"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

print(
    overall_rules_filtered[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20)
)

                  antecedents                   consequents   support  \
0                 PASTA SAUCE             DRY NOODLES/PASTA  0.025060   
1           DRY NOODLES/PASTA                   PASTA SAUCE  0.025060   
2                  DELI MEATS                       CHEESES  0.027826   
3                     CHEESES                    DELI MEATS  0.027826   
4                 PEPPERS-ALL                        ONIONS  0.013566   
5                      ONIONS                   PEPPERS-ALL  0.013566   
6                PAPER TOWELS                  BATH TISSUES  0.014604   
7                BATH TISSUES                  PAPER TOWELS  0.014604   
8            VEGETABLES SALAD                      TOMATOES  0.016043   
9                    TOMATOES              VEGETABLES SALAD  0.016043   
10    VEGETABLES - ALL OTHERS                   PEPPERS-ALL  0.012988   
11                PEPPERS-ALL       VEGETABLES - ALL OTHERS  0.012988   
12                    CARROTS       VEGETABLES - AL

In [17]:
# Save the results as a table named overall_rules
overall_rules_filtered.to_csv(
    r"C:\Users\15195\Desktop\overall_rules.csv",
    index=False
)

print("Saved overall_rules.csv ")

Saved overall_rules.csv 


In [18]:
# Create the segment_basket_items.csv table

segment_basket_items = (
    basket_source[
        [
            "cluster",
            "household_key",
            "BASKET_ID",
            "COMMODITY_DESC"
        ]
    ]
    .drop_duplicates()
)

print(segment_basket_items.head())


   cluster  household_key    BASKET_ID               COMMODITY_DESC
0        1           2375  26984851472                     POTATOES
1        1           2375  26984851472                       ONIONS
2        1           2375  26984851472      VEGETABLES - ALL OTHERS
3        1           2375  26984851472               TROPICAL FRUIT
4        1           2375  26984851472  ORGANICS FRUIT & VEGETABLES


In [19]:
# Save the segment_basket_items.csv table.
segment_basket_items.to_csv(
    r"C:\Users\15195\Desktop\segment_basket_items.csv",
    index=False
)

print("Saved segment_basket_items.csv")

Saved segment_basket_items.csv


In [20]:
# Verify the number of baskets in each cluster

segment_basket_counts = (
    segment_basket_items
    .groupby("cluster")["BASKET_ID"]
    .nunique()
    .reset_index(name="basket_count")
)

print(segment_basket_counts)


   cluster  basket_count
0        0         79089
1        1        181887
2        2         14563


In [21]:
# Save the table segment_basket_counts
segment_basket_counts.to_csv(
    r"C:\Users\15195\Desktop\segment_basket_counts.csv",
    index=False
)

print("Saved segment_basket_counts.csv")

Saved segment_basket_counts.csv


In [22]:
# Run Apriori separately for each cluster
all_segment_itemsets = []
all_segment_rules = []

clusters = sorted(
    segment_basket_items["cluster"]
    .dropna()
    .unique()
)

for cluster_id in clusters:

    print(f"\nProcessing cluster {cluster_id}")

    segment_data = segment_basket_items[
        segment_basket_items["cluster"] == cluster_id
    ].copy()

    segment_total_baskets = (
        segment_data["BASKET_ID"]
        .nunique()
    )

    print("Number of baskets:", segment_total_baskets)

    if segment_total_baskets < 100:
        print(
            f"Cluster {cluster_id} has fewer than 100 baskets and was skipped."
        )
        continue

    segment_commodity_frequency = (
        segment_data
        .groupby("COMMODITY_DESC")["BASKET_ID"]
        .nunique()
        .sort_values(ascending=False)
    )

    segment_minimum_count = max(
        10,
        int(segment_total_baskets * 0.01)
    )

    segment_eligible_commodities = (
        segment_commodity_frequency[
            segment_commodity_frequency >= segment_minimum_count
        ]
        .index
    )

    segment_data_filtered = segment_data[
        segment_data["COMMODITY_DESC"]
        .isin(segment_eligible_commodities)
    ].copy()

    segment_top_commodities = (
        segment_data_filtered["COMMODITY_DESC"]
        .value_counts()
        .head(100)
        .index
    )

    segment_data_filtered = segment_data_filtered[
        segment_data_filtered["COMMODITY_DESC"]
        .isin(segment_top_commodities)
    ].copy()

    segment_matrix = (
        segment_data_filtered
        .assign(present=True)
        .pivot_table(
            index="BASKET_ID",
            columns="COMMODITY_DESC",
            values="present",
            aggfunc="max",
            fill_value=False
        )
        .astype(bool)
    )

    print("Segment matrix shape:", segment_matrix.shape)

    segment_itemsets = apriori(
        segment_matrix,
        min_support=0.01,
        use_colnames=True,
        max_len=3
    )

    if segment_itemsets.empty:
        print(
            f"No frequent itemsets found for cluster {cluster_id}."
        )
        continue

    segment_itemsets["cluster"] = cluster_id
    segment_itemsets["segment_basket_count"] = (
        segment_total_baskets
    )

    all_segment_itemsets.append(
        segment_itemsets.copy()
    )

    segment_rules = association_rules(
        segment_itemsets.drop(
            columns=[
                "cluster",
                "segment_basket_count"
            ],
            errors="ignore"
        ),
        metric="confidence",
        min_threshold=0.10
    )

    if segment_rules.empty:
        print(
            f"No association rules found for cluster {cluster_id}."
        )
        continue

    segment_rules = segment_rules[
        (segment_rules["support"] >= 0.005) &
        (segment_rules["confidence"] >= 0.20) &
        (segment_rules["lift"] > 1.00)
    ].copy()

    segment_rules = segment_rules[
        (segment_rules["antecedents"].apply(len) == 1) &
        (segment_rules["consequents"].apply(len) == 1)
    ].copy()

    if segment_rules.empty:
        print(
            f"No rules passed the filters for cluster {cluster_id}."
        )
        continue

    segment_rules["cluster"] = cluster_id
    segment_rules["segment_basket_count"] = (
        segment_total_baskets
    )

    all_segment_rules.append(
        segment_rules.copy()
    )

    print(
        f"Cluster {cluster_id}: "
        f"{len(segment_rules)} filtered rules"
    )


Processing cluster 0
Number of baskets: 79089
Segment matrix shape: (73708, 100)
Cluster 0: 1277 filtered rules

Processing cluster 1
Number of baskets: 181887
Segment matrix shape: (169760, 100)
Cluster 1: 1203 filtered rules

Processing cluster 2
Number of baskets: 14563
Segment matrix shape: (13303, 100)
Cluster 2: 242 filtered rules


In [23]:
# Save Segment Frequent Itemsets
if all_segment_itemsets:

    segment_frequent_itemsets = pd.concat(
        all_segment_itemsets,
        ignore_index=True
    )

    segment_frequent_itemsets["itemsets"] = (
        segment_frequent_itemsets["itemsets"]
        .apply(
            lambda x: " | ".join(
                sorted(list(x))
            )
        )
    )

    segment_frequent_itemsets.to_csv(
        r"C:\Users\15195\Desktop\segment_frequent_itemsets.csv",
        index=False
    )

    print("Saved segment_frequent_itemsets.csv")

else:
    print("No segment frequent itemsets were generated")

Saved segment_frequent_itemsets.csv


In [24]:
# Save the segment_association_rules.csv table
if all_segment_rules:

    segment_association_rules = pd.concat(
        all_segment_rules,
        ignore_index=True
    )

    segment_association_rules["antecedents"] = (
        segment_association_rules["antecedents"]
        .apply(
            lambda x: " | ".join(
                sorted(list(x))
            )
        )
    )

    segment_association_rules["consequents"] = (
        segment_association_rules["consequents"]
        .apply(
            lambda x: " | ".join(
                sorted(list(x))
            )
        )
    )

    segment_association_rules = (
        segment_association_rules
        .sort_values(
            by=[
                "cluster",
                "lift",
                "confidence",
                "support"
            ],
            ascending=[
                True,
                False,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    segment_association_rules.to_csv(
        r"C:\Users\15195\Desktop\segment_association_rules.csv",
        index=False
    )

    print("Saved segment_association_rules.csv")

    print(
        segment_association_rules[
            [
                "cluster",
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ].head(30)
    )

else:
    print("No segment association rules were generated.")

Saved segment_association_rules.csv
    cluster                antecedents                consequents   support  \
0         0                PASTA SAUCE          DRY NOODLES/PASTA  0.022711   
1         0          DRY NOODLES/PASTA                PASTA SAUCE  0.022711   
2         0                    CHEESES                 DELI MEATS  0.030662   
3         0                 DELI MEATS                    CHEESES  0.030662   
4         0               PAPER TOWELS               BATH TISSUES  0.013676   
5         0               BATH TISSUES               PAPER TOWELS  0.013676   
6         0                PEPPERS-ALL                     ONIONS  0.012916   
7         0                     ONIONS                PEPPERS-ALL  0.012916   
8         0  BEANS - CANNED GLASS & MW  VEGETABLES - SHELF STABLE  0.017678   
9         0  VEGETABLES - SHELF STABLE  BEANS - CANNED GLASS & MW  0.017678   
10        0           DRY SAUCES/GRAVY  VEGETABLES - SHELF STABLE  0.012495   
11        0     

In [25]:
# Extract the top 20 rules for each cluster
if all_segment_rules:

    top_segment_rules = (
        segment_association_rules
        .sort_values(
            by=[
                "cluster",
                "lift",
                "confidence",
                "support"
            ],
            ascending=[
                True,
                False,
                False,
                False
            ]
        )
        .groupby(
            "cluster",
            group_keys=False
        )
        .head(20)
        .reset_index(drop=True)
    )

    print(
        top_segment_rules[
            [
                "cluster",
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ]
    )

    top_segment_rules.to_csv(
        r"C:\Users\15195\Desktop\top20_rules_by_segment.csv",
        index=False
    )

    print("Saved top20_rules_by_segment.csv. ")

    cluster                antecedents                   consequents  \
0         0                PASTA SAUCE             DRY NOODLES/PASTA   
1         0          DRY NOODLES/PASTA                   PASTA SAUCE   
2         0                    CHEESES                    DELI MEATS   
3         0                 DELI MEATS                       CHEESES   
4         0               PAPER TOWELS                  BATH TISSUES   
5         0               BATH TISSUES                  PAPER TOWELS   
6         0                PEPPERS-ALL                        ONIONS   
7         0                     ONIONS                   PEPPERS-ALL   
8         0  BEANS - CANNED GLASS & MW     VEGETABLES - SHELF STABLE   
9         0  VEGETABLES - SHELF STABLE     BEANS - CANNED GLASS & MW   
10        0           DRY SAUCES/GRAVY     VEGETABLES - SHELF STABLE   
11        0           VEGETABLES SALAD                      TOMATOES   
12        0                   TOMATOES              VEGETABLES S

In [26]:
# Read category_summary.csv
category_summary = pd.read_csv(
    r"C:\Users\15195\Desktop\coding part\EDA part\category_summary.csv"
)

print(category_summary.columns.tolist())
print(category_summary.head())

['COMMODITY_DESC', 'total_sales', 'total_quantity', 'basket_count', 'household_count', 'product_count', 'department_count', 'category_sales_share', 'category_penetration', 'average_sales_per_basket']
        COMMODITY_DESC  total_sales  total_quantity  basket_count  \
0    COUPON/MISC ITEMS    639878.56       257218037         27705   
1          SOFT DRINKS    327647.30          160637         71699   
2                 BEEF    312103.22           65576         36733   
3  FLUID MILK PRODUCTS    205356.05          116192         69278   
4               CHEESE    189528.18           96402         46898   

   household_count  product_count  department_count  category_sales_share  \
0             1985            128                 9              0.079414   
1             2405           1704                 1              0.040664   
2             2238           1109                 1              0.038735   
3             2421            455                 1              0.025486   


In [27]:
# Standardize the format of category names
category_summary["COMMODITY_DESC"] = (
    category_summary["COMMODITY_DESC"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [28]:
# Select the Category Management metric
# First, check what the column labeled “Sales Percentage” is called
if "category_sales_share" in category_summary.columns:
    sales_share_column = "category_sales_share"

elif "sales_share" in category_summary.columns:
    sales_share_column = "sales_share"

else:
    raise ValueError(
        "Neither category_sales_share nor sales_share exists."
    )

In [29]:
# Extract indicators、
category_metrics = category_summary[
    [
        "COMMODITY_DESC",
        "total_sales",
        "basket_count",
        "household_count",
        "category_penetration",
        sales_share_column
    ]
].copy()

category_metrics = category_metrics.rename(
    columns={
        sales_share_column: "category_sales_share"
    }
)

print(category_metrics.head())

        COMMODITY_DESC  total_sales  basket_count  household_count  \
0    COUPON/MISC ITEMS    639878.56         27705             1985   
1          SOFT DRINKS    327647.30         71699             2405   
2                 BEEF    312103.22         36733             2238   
3  FLUID MILK PRODUCTS    205356.05         69278             2421   
4               CHEESE    189528.18         46898             2350   

   category_penetration  category_sales_share  
0                0.7940              0.079414  
1                0.9620              0.040664  
2                0.8952              0.038735  
3                0.9684              0.025486  
4                0.9400              0.023522  


In [30]:
# Merge the Category metrics into Segment Rules
# This step is intended to run only when successful segment rules have been generated
if all_segment_rules:

    category_rule_summary = (
        segment_association_rules
        .merge(
            category_metrics.add_prefix("antecedent_"),
            left_on="antecedents",
            right_on="antecedent_COMMODITY_DESC",
            how="left"
        )
    )

    category_rule_summary = (
        category_rule_summary
        .merge(
            category_metrics.add_prefix("consequent_"),
            left_on="consequents",
            right_on="consequent_COMMODITY_DESC",
            how="left"
        )
    )

In [31]:
# Keep the key columns
if all_segment_rules:

    category_rule_summary = category_rule_summary[
        [
            "cluster",
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift",

            "antecedent_total_sales",
            "antecedent_basket_count",
            "antecedent_household_count",
            "antecedent_category_penetration",
            "antecedent_category_sales_share",

            "consequent_total_sales",
            "consequent_basket_count",
            "consequent_household_count",
            "consequent_category_penetration",
            "consequent_category_sales_share"
        ]
    ]

    print(category_rule_summary.head())

   cluster        antecedents        consequents   support  confidence  \
0        0        PASTA SAUCE  DRY NOODLES/PASTA  0.022711    0.536367   
1        0  DRY NOODLES/PASTA        PASTA SAUCE  0.022711    0.442039   
2        0            CHEESES         DELI MEATS  0.030662    0.599788   
3        0         DELI MEATS            CHEESES  0.030662    0.451098   
4        0       PAPER TOWELS       BATH TISSUES  0.013676    0.379233   

        lift  antecedent_total_sales  antecedent_basket_count  \
0  10.439531                33993.40                    11725   
1  10.439531                23361.48                    14021   
2   8.824182                52299.00                    11331   
3   8.824182               106539.94                    17122   
4   8.645985                30254.54                    10178   

   antecedent_household_count  antecedent_category_penetration  \
0                        1816                           0.7264   
1                        1962   

In [32]:
# Save the form
if all_segment_rules:

    category_rule_summary.to_csv(
        r"C:\Users\15195\Desktop\category_rule_summary.csv",
        index=False
    )

    print("Saved category_rule_summary.csv")

Saved category_rule_summary.csv
